# Create files mapping FACS well and plate to basic allele metadata

This script takes cherry picking and allele info files as input and produces a table relating FACS well and plate IDs to allele IDs and symbols. 

In [1]:
import pandas as pd
import os

In [8]:
dualipa_inputs = "../1_inputs"
allele_collection_inputs = "../../../1_allele_collection/1_inputs"
dualipa_outputs = "../3_outputs"

In [ ]:
# Download raw FACS data from Zenodo
bash_file = f"{dualipa_inputs}/download_raw_FACS.sh"
os.system(f"bash {base_file}")

In [6]:
f"{allele_collection_inputs}/VarChamp_Consolidated_allele_collection.tsv"

'../../1_allele_collection/1_inputs/VarChamp_Consolidated_allele_collection.tsv'

In [20]:
# REPLACE THIS WITH TONG'S FILE ONCE HE ADD MUTATION ID
allele_info_df = pd.read_table(f"{allele_collection_inputs}/VarChamp_Consolidated_allele_collection.tsv")
other_cols = [i for i in allele_info_df.columns if "mutation_id_" not in i]
mut_cols = [i for i in allele_info_df.columns if "mutation_id_" in i]

In [21]:
allele_info_df = allele_info_df.melt(id_vars=other_cols, 
                                     value_vars=mut_cols, 
                                     var_name='mut_type', 
                                     value_name='mut_id')

In [22]:
allele_info_df

,orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,entry_plate,entry_well,collection,sequence,mut_type,mut_id
0,2,GBA,160G>C,Val54Leu,ENSG00000177628,1,155240033,NC_000001.11:155240033:C:G,ALE0000584,CCSBVarC000001,GDEhDisVCh_40054,F12,RC4,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,mutation_id_rc4,6.0
1,2,GBA,670C>T,Leu224Phe,ENSG00000177628,1,155238225,NC_000001.11:155238225:G:A,ALE00000002,CCSBVarC000002,GDEhDisVCh_40054,C12,RC4,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,mutation_id_rc4,73.0
2,2,GBA,887G>A,Arg296Gln,ENSG00000177628,1,155237453,NC_000001.11:155237453:C:T,ALE00000003,CCSBVarC000003,GDEhDisVCh_40054,D12,RC4,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,mutation_id_rc4,113.0
3,2,GBA,1448T>C,Leu483Pro,ENSG00000177628,1,155235252,NC_000001.11:155235252:A:G,ALE00000004,CCSBVarC000004,GDEhDisVCh_40054,E12,RC4,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,mutation_id_rc4,231.0
4,25,SLC9A3R1,328C>G,Leu110Val,ENSG00000109062,17,74749174,NC_000017.11:74749174:C:G,ALE00000005,CCSBVarC000005,GDEhDisVCh_40045,E04,RC4,ATGAGCGCGGACGCAGCGGCCGGGGCGCCCCTGCCCCGGCTCTGCT...,mutation_id_rc4,252.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25459,8999,TTR,355G>A,Asp119Asn,ENSG00000118271,18,31598586,NC_000018.10:31598586:G:A,ALE00008468,CCSBVarC008481,NaN,NaN,Edgotyping3,ATGGCTTCTCATCGTCTGCTCCTCCTCTGCCTTGCTGGACTGGTAT...,mutation_id_edgotyping3,NaN
25460,7323,PMP22,152A>G,His51Arg,ENSG00000109099,17,15259120,NC_000017.11:15259120:T:C,ALE00008469,CCSBVarC008466,NaN,NaN,Edgotyping3,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,mutation_id_edgotyping3,NaN
25461,2651,CLN6,89G>C,Gly30Ala,ENSG00000128973,15,68218645,NC_000015.10:68218645:C:G,ALE00008470,CCSBVarC008436,NaN,NaN,Edgotyping3,ATGGAGGCGACGCGGAGGCGGCAGCACCTGGGAGCGACGGGCGGCC...,mutation_id_edgotyping3,NaN
25462,7323,PMP22,403G>A,Ala135Thr,ENSG00000109099,17,15230997,NC_000017.11:15230997:C:T,ALE00008471,CCSBVarC008467,NaN,NaN,Edgotyping3,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,mutation_id_edgotyping3,NaN


Vidal lab ran experiment once and analyzed the data. Some alleles didn't pass the qa/qc thresholds or got no data. Batch5 is is a re-run for all alleles that failed in the first set of batches. Here, we process everything together from scratch. An allele should always be compared to the wt on the same batch and plate. The summary stats can then be compared across batches. 

In [11]:
# Read in cherry picking files for both the initial batches and the second round
cp_col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']

pdest_layout_initial_batches_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP.csv', index_col=0)[cp_col]
pdest_layout_b5_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP_batch5.csv', index_col=0)[cp_col]

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_initial_batches_df])

In [23]:
pdest_layout_df = pdest_layout_df.merge(allele_info_df, on=['orf_id', 'mut_id'], how='left')

In [ ]:
# aa change, nt change, symbol

In [ ]:
# Use mutation and orf ID to add nt change, aa change, and clinvar label
# REDO WITH TONG'S UPDATED ALLELE INFO FILE
pdest_layout_df['nt_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][0] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
pdest_layout_df['aa_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][1] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

# REDO WITH CHLOE'S UPDATED CLINVAR ANNOTATIONS
pdest_layout_df['clinical_significance'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][2] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

In [ ]:
pdest_layout_df.to_csv(f'{dualipa_outputs}/pdest_experimental_layout.csv', index=None)